# 01 — Data and chronological feature engineering
Boréal Marché. Notebook updated with AI assistance; see docs/AI_USE.md.
The executable implementation lives in src/ingest.py and src/features.py. This notebook inspects
its saved outputs without creating a competing train/test protocol. Run `make data features` first.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
def report(name):
    return json.loads((ROOT / "reports/current_run" / name).read_text())

In [ ]:
from src.contracts import FEATURE_NAMES, ENGINEERED_FEATURE_NAMES
transactions = pd.read_parquet(ROOT / "data/processed/transactions.parquet")
X = pd.read_parquet(ROOT / "data/processed/X.parquet")
snapshots = pd.read_parquet(ROOT / "data/processed/snapshots.parquet")
assert list(X.columns) == ENGINEERED_FEATURE_NAMES
assert len(FEATURE_NAMES) == 12 and len(ENGINEERED_FEATURE_NAMES) == 25
print(f"Clean event rows: {len(transactions):,}; engineered columns: {X.shape[1]}; serving columns: {len(FEATURE_NAMES)}")
display(transactions.head())

## Source limitations
UCI Online Retail II is historical UK giftware data; the Québec grocery company is fictional.
Amounts remain GBP. Returns remain events while positive purchases drive the purchase features.
All 25 engineered features are documented in governance/FEATURE_INVENTORY.md; 13 are excluded
from serving after the original June minimisation audit. Country fields remain offline for auditing.

In [ ]:
display(snapshots.groupby(["split", "cutoff", "label_end"], sort=False).size().rename("rows").reset_index())
display(X.describe().T)
print("Model inputs:", FEATURE_NAMES)
print("Excluded from serving:", [c for c in X if c not in FEATURE_NAMES])

## Structural defaults and meaning
Gap standard deviation is zero with fewer than two observed gaps. One-time buyer means exactly
one positive invoice. Single-item ratio counts invoices with one line, not units. Entropy is
base-2 Shannon entropy of first-token category spend shares. Return rate uses negative-unit share
of all absolute event units, not the fraction of originally sold units subsequently returned.
These are definitions, not reliable evidence of behaviour when history is sparse.

In [ ]:
display(X[FEATURE_NAMES].isna().sum().rename("missing"))
print("All features precede each cutoff; labels cover the following 90 days.")